# Predictive Maintenance — AI4I 2020
Goal: predict `Machine failure` from sensor readings.

**Before running:** download the dataset to `data/ai4i2020.csv` (see `data/README.md`).

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath(".."))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (classification_report, ConfusionMatrixDisplay,
                             PrecisionRecallDisplay, RocCurveDisplay)

from src.data_preprocessing import (load_data, clean_data, add_features, build_preprocessor,
                                    FEATURES, TARGET, NUMERIC)

sns.set_theme(style="whitegrid")
DATA_PATH = "../data/ai4i2020.csv"

## 1. Load & inspect

In [ ]:
raw = load_data(DATA_PATH)
print(raw.shape)
raw.head()

In [ ]:
raw.info()
raw.describe().T

## 2. Target & failure modes

In [ ]:
modes = ["TWF", "HDF", "PWF", "OSF", "RNF"]
print(raw["Machine failure"].value_counts(normalize=True).round(4))
print()
print(raw[modes].sum())
# Machine failure is (almost) the OR of the individual modes -> they would leak the target

## 3. Clean & engineer features

In [ ]:
df = add_features(clean_data(raw))
df.head()

## 4. EDA

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(18, 7))
for ax, col in zip(axes.ravel(), NUMERIC):
    sns.kdeplot(data=df, x=col, hue=TARGET, common_norm=False, fill=True, ax=ax)
    ax.set_title(col)
plt.tight_layout(); plt.show()

In [ ]:
print(df.groupby("type")[TARGET].mean().round(4))
plt.figure(figsize=(8, 6))
sns.heatmap(df[NUMERIC + [TARGET]].corr(), annot=True, fmt=".2f", cmap="coolwarm")
plt.title("Correlation matrix"); plt.show()

## 5. Train / test split

In [ ]:
X, y = df[FEATURES], df[TARGET].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)
print(X_train.shape, X_test.shape, y_train.mean().round(4), y_test.mean().round(4))

## 6. Compare models (5-fold stratified CV, PR-AUC)

In [ ]:
models = {
    "logistic_regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "random_forest": RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                            n_jobs=-1, random_state=42),
    "gradient_boosting": GradientBoostingClassifier(random_state=42),
}
cv = StratifiedKFold(5, shuffle=True, random_state=42)
for name, clf in models.items():
    pipe = Pipeline([("prep", build_preprocessor()), ("clf", clf)])
    s = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="average_precision", n_jobs=-1)
    print(f"{name:22s} PR-AUC = {s.mean():.3f} ± {s.std():.3f}")

## 7. Final evaluation on the held-out test set

In [ ]:
best = Pipeline([("prep", build_preprocessor()),
                 ("clf", RandomForestClassifier(n_estimators=300, class_weight="balanced_subsample",
                                                n_jobs=-1, random_state=42))])
best.fit(X_train, y_train)
print(classification_report(y_test, best.predict(X_test), target_names=["No failure", "Failure"]))

fig, axes = plt.subplots(1, 3, figsize=(17, 4.5))
ConfusionMatrixDisplay.from_estimator(best, X_test, y_test, ax=axes[0], colorbar=False)
PrecisionRecallDisplay.from_estimator(best, X_test, y_test, ax=axes[1])
RocCurveDisplay.from_estimator(best, X_test, y_test, ax=axes[2])
plt.tight_layout(); plt.show()

## 8. Feature importance

In [ ]:
prep, clf = best.named_steps["prep"], best.named_steps["clf"]
names = prep.get_feature_names_out()
imp = pd.Series(clf.feature_importances_, index=names).sort_values()
imp.plot.barh(figsize=(8, 6), title="Random Forest feature importance"); plt.show()

## 9. Takeaways
- Failures are rare (~3.4%), so accuracy is misleading — use PR-AUC / recall / F1.
- Engineered features (`temp_diff`, `power_w`, `wear_torque`) map onto the physical failure mechanisms and rank highly.
- Next steps: tune the decision threshold to business costs, add SHAP, predict failure *mode*.